# 02 · Filtro de Minas Gerais e montagem da base da RMBH

**Objetivo:** filtrar os municípios de Minas Gerais com SQL (DuckDB), conferir duplicidades, montar a lista dos 34 municípios da Região Metropolitana de Belo Horizonte e salvar as bases tratadas em `data/processed/`, usadas pelos notebooks seguintes.

> Parte 2 de 6 do projeto *BH e Região sob a perspectiva dos dados*. Veja o [README](../README.md) para a visão geral.

In [1]:
# Configuração: funciona no computador (a partir da pasta notebooks/) e no Google Colab
import os
import subprocess
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    repo = Path('/content/ips-bh-rmbh')
    if not repo.exists():
        subprocess.run(['git', 'clone', '-q', 'https://github.com/millebns/ips-bh-rmbh.git', str(repo)], check=True)
    os.chdir(repo / 'notebooks')

sys.path.insert(0, str(Path.cwd().parent / 'src'))

In [2]:
import duckdb
import pandas as pd

from ips_rmbh.config import CAMINHO_MG_2026, CAMINHO_RMBH_2026, PASTA_TRATADOS
from ips_rmbh.dados import carregar_edicao

df24 = carregar_edicao('2024')
df25 = carregar_edicao('2025')
df26 = carregar_edicao('2026')

# 1. Filtro UF = MG

In [3]:
filtroUF24 = """
SELECT *
FROM df24
WHERE UF = 'MG'
"""

filtroUF25 = """
SELECT *
FROM df25
WHERE UF = 'MG'
"""

filtroUF26 = """
SELECT *
FROM df26
WHERE UF = 'MG'
"""

In [4]:
df24_mg = duckdb.query(filtroUF24).df()
df25_mg = duckdb.query(filtroUF25).df()
df26_mg = duckdb.query(filtroUF26).df()

In [5]:
for ano, d in [('2024', df24_mg), ('2025', df25_mg), ('2026', df26_mg)]:
    print(ano, d.shape)

2024 (853, 74)
2025 (853, 78)
2026 (853, 78)


In [6]:
assert df24_mg['Município'].duplicated().sum() == 0

In [7]:
assert df25_mg['Município'].duplicated().sum() == 0

In [8]:
assert df26_mg['Município'].duplicated().sum() == 0

# 2. Região Metropolitana de Belo Horizonte (RMBH)

1. Baldim
2. Belo Horizonte
3. Betim
4. Brumadinho
5. Caeté
6. Capim Branco
7. Confins
8. Contagem
9. Esmeraldas
10. Florestal
11. Ibirité
12. Igarapé
13. Itaguara
14. Itatiaiuçu
15. Jaboticatubas
16. Juatuba
17. Lagoa Santa
18. Mário Campos
19. Mateus Leme
20. Matozinhos
21. Nova Lima
22. Nova União
23. Pedro Leopoldo
24. Raposos
25. Ribeirão das Neves
26. Rio Acima
27. Rio Manso
28. Sabará
29. Santa Luzia
30. São Joaquim de Bicas
31. São José da Lapa
32. Sarzedo
33. Taquaraçu de Minas
34. Vespasiano

In [9]:
# A lista fica em src/ips_rmbh/config.py para ser reutilizada pelos outros notebooks
from ips_rmbh.config import MUNICIPIOS_RMBH as municipios_rmbh

In [10]:
df_rmbh = df26_mg[df26_mg['Município'].isin(municipios_rmbh)].copy()
df_rmbh

,Município,UF,Índice de Progresso Social,ÁREA km²,POPULAÇÃO 2025,PIB PER CAPITA,Necessidades Humanas Básicas,Fundamentos do Bem-estar,Oportunidades,Nutrição e Cuidados Médicos Básicos,...,Praças e Parques em Áreas Urbanas,Famílias em Situação de Rua,Paridade de Gênero na Câmara Municipal,Paridade de Negros na Câmara Municipal,Violência Contra Indígenas,Violência Contra Mulheres,Violência Contra Negros,Empregados com Ensino Superior,Mulheres Empregadas com Ensino Superior,Nota Mediana no Enem
0,Nova Lima,MG,71.22,429.3,120959.0,102907.19,78.67,74.34,60.66,75.89,...,8.30,48.39,0.13,0.95,NaN,164.07,80.35,214.05,223.98,590.38
4,Confins,MG,69.67,42.5,7758.0,208432.20,74.84,73.08,61.08,82.87,...,0.51,15.00,0.22,0.00,NaN,290.68,0.00,225.05,214.05,483.24
5,Belo Horizonte,MG,69.66,331.4,2415872.0,53892.62,79.15,74.87,54.94,74.62,...,3.35,432.01,0.59,0.74,78.04,281.02,186.69,240.92,265.37,591.52
12,Sarzedo,MG,68.47,62.1,39974.0,41089.76,83.21,73.42,48.79,79.25,...,0.45,19.34,0.89,0.76,133.33,180.06,60.02,50.37,63.09,533.74
28,Contagem,MG,67.57,194.7,651718.0,69190.04,78.71,72.99,51.00,73.46,...,1.48,86.05,0.56,0.75,50.47,213.96,69.00,72.28,74.80,562.66
78,Rio Acima,MG,66.04,228.4,10698.0,24337.45,83.38,69.12,45.61,78.43,...,0.25,23.99,0.44,0.89,NaN,385.49,177.73,54.50,73.27,472.28
80,Florestal,MG,65.98,194.2,8469.0,21597.47,80.12,74.30,43.53,73.50,...,2.71,1.53,0.67,0.58,NaN,106.39,88.89,25.70,36.08,608.03
89,Igarapé,MG,65.77,111.0,49120.0,28538.76,80.55,74.21,42.54,80.85,...,2.19,35.27,0.31,0.25,60.61,154.29,56.92,37.36,53.07,539.69
99,Ibirité,MG,65.43,72.4,179582.0,18017.57,79.28,71.20,45.81,75.64,...,0.58,21.97,0.00,0.47,100.00,261.43,36.79,28.62,38.56,548.00
103,Caeté,MG,65.39,542.5,39775.0,25245.28,79.80,71.42,44.96,74.26,...,0.42,8.22,0.15,0.98,NaN,124.14,4.23,33.91,49.47,555.69


In [11]:
print('Municípios não encontrados:', set(municipios_rmbh) - set(df_rmbh['Município']))
assert len(df_rmbh) == 34

Municípios não encontrados: set()


In [12]:
nulos = df_rmbh.isna().sum()
print(nulos[nulos > 0])
df_rmbh.dtypes.value_counts()

Violência Contra Indígenas    16
dtype: int64


float64    76
str         2
Name: count, dtype: int64

In [13]:
(df_rmbh[['Município', 'POPULAÇÃO 2025', 'PIB PER CAPITA']]
 .sort_values('POPULAÇÃO 2025', ascending=False)
 .reset_index(drop=True))

,Município,POPULAÇÃO 2025,PIB PER CAPITA
0,Belo Horizonte,2415872.0,53892.62
1,Contagem,651718.0,69190.04
2,Betim,431433.0,121952.48
3,Ribeirão das Neves,346971.0,18262.90
4,Santa Luzia,230382.0,28683.22
5,Ibirité,179582.0,18017.57
6,Vespasiano,138583.0,26114.54
7,Sabará,134576.0,28513.20
8,Nova Lima,120959.0,102907.19
9,Esmeraldas,93103.0,14120.82


Os 34 municípios foram encontrados. Os portes são muito diferentes: BH tem cerca de 2,4 milhões de habitantes, e vários municípios têm menos de 10 mil.

# 3. Salvando as bases tratadas

In [14]:
PASTA_TRATADOS.mkdir(parents=True, exist_ok=True)
df26_mg.to_csv(CAMINHO_MG_2026, index=False)
df_rmbh.to_csv(CAMINHO_RMBH_2026, index=False)
print(f'{CAMINHO_MG_2026.name}: {df26_mg.shape} | {CAMINHO_RMBH_2026.name}: {df_rmbh.shape}')

ips_2026_mg.csv: (853, 78) | ips_2026_rmbh.csv: (34, 78)


**Próximo notebook:** [03 · BH no ranking da RMBH](03_ranking_ips_rmbh.ipynb)